In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

print(torch.__version__)

2.11.0+cpu


In [2]:
# Creating a tensor
x = torch.tensor([10, 20, 30, 40])

print(x)
print("Shape:", x.shape)
print("Data type:", x.dtype)

tensor([10, 20, 30, 40])
Shape: torch.Size([4])
Data type: torch.int64


In [3]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X = torch.tensor([
    [2, 60, 50],
    [4, 70, 60],
    [6, 80, 70],
    [8, 90, 85],
    [3, 65, 55],
    [7, 85, 80]
], dtype=torch.float32)

y = torch.tensor([
    0, 0, 1, 1, 0, 1
], dtype=torch.float32).reshape(-1, 1)

print("Input data:")
print(X)

print("\nTarget:")
print(y)

Input data:
tensor([[ 2., 60., 50.],
        [ 4., 70., 60.],
        [ 6., 80., 70.],
        [ 8., 90., 85.],
        [ 3., 65., 55.],
        [ 7., 85., 80.]])

Target:
tensor([[0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.]])


In [4]:
dataset = TensorDataset(X, y)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

for features, labels in dataloader:
    print("Features:")
    print(features)
    print("Labels:")
    print(labels)
    print("---")

Features:
tensor([[ 2., 60., 50.],
        [ 8., 90., 85.]])
Labels:
tensor([[0.],
        [1.]])
---
Features:
tensor([[ 7., 85., 80.],
        [ 4., 70., 60.]])
Labels:
tensor([[1.],
        [0.]])
---
Features:
tensor([[ 6., 80., 70.],
        [ 3., 65., 55.]])
Labels:
tensor([[1.],
        [0.]])
---


In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)


model = NeuralNetwork()

print(model)

NeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [6]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss Function:", criterion)
print("Optimizer:", optimizer)

Loss Function: BCELoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [7]:
epochs = 100

for epoch in range(epochs):
    total_loss = 0

    for features, labels in dataloader:
        # Forward pass
        predictions = model(features)

        # Calculate loss
        loss = criterion(predictions, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {total_loss:.4f}")

Epoch [10/100], Loss: 5.8399
Epoch [20/100], Loss: 2.4742
Epoch [30/100], Loss: 2.0639
Epoch [40/100], Loss: 2.0279
Epoch [50/100], Loss: 1.9932
Epoch [60/100], Loss: 1.9634
Epoch [70/100], Loss: 1.9391
Epoch [80/100], Loss: 1.8837
Epoch [90/100], Loss: 1.8619
Epoch [100/100], Loss: 1.8047


In [8]:
correct = 0
total = 0

with torch.no_grad():
    for features, labels in dataloader:
        predictions = model(features)

        predicted_labels = (predictions >= 0.5).float()

        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 50.00%


In [9]:
with torch.no_grad():
    predictions = model(X)

    predicted_labels = (predictions >= 0.5).float()

    print("Actual Labels:")
    print(y)

    print("\nPredicted Probabilities:")
    print(predictions)

    print("\nPredicted Labels:")
    print(predicted_labels)

Actual Labels:
tensor([[0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.]])

Predicted Probabilities:
tensor([[0.5672],
        [0.5990],
        [0.6428],
        [0.7664],
        [0.5765],
        [0.7493]])

Predicted Labels:
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.]])


In [10]:
X_normalized = X.clone()

X_normalized[:, 0] = X[:, 0] / 10
X_normalized[:, 1] = X[:, 1] / 100
X_normalized[:, 2] = X[:, 2] / 100

print(X_normalized)

tensor([[0.2000, 0.6000, 0.5000],
        [0.4000, 0.7000, 0.6000],
        [0.6000, 0.8000, 0.7000],
        [0.8000, 0.9000, 0.8500],
        [0.3000, 0.6500, 0.5500],
        [0.7000, 0.8500, 0.8000]])


In [11]:
model = NeuralNetwork()

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

normalized_dataset = TensorDataset(X_normalized, y)

normalized_dataloader = DataLoader(
    normalized_dataset,
    batch_size=2,
    shuffle=True
)

epochs = 200

for epoch in range(epochs):
    total_loss = 0

    for features, labels in normalized_dataloader:
        predictions = model(features)

        loss = criterion(predictions, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {total_loss:.4f}")

Epoch [20/200], Loss: 2.0431
Epoch [40/200], Loss: 1.6655
Epoch [60/200], Loss: 1.1356
Epoch [80/200], Loss: 0.7873
Epoch [100/200], Loss: 0.5959
Epoch [120/200], Loss: 0.4707
Epoch [140/200], Loss: 0.3427
Epoch [160/200], Loss: 0.2816
Epoch [180/200], Loss: 0.2334
Epoch [200/200], Loss: 0.1956


In [12]:
correct = 0
total = 0

with torch.no_grad():
    for features, labels in normalized_dataloader:
        predictions = model(features)
        predicted_labels = (predictions >= 0.5).float()

        total += labels.size(0)
        correct += (predicted_labels == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 100.00%


In [13]:
with torch.no_grad():
    predictions = model(X_normalized)
    predicted_labels = (predictions >= 0.5).float()

print("Actual Labels:")
print(y)

print("\nPredicted Probabilities:")
print(predictions)

print("\nPredicted Labels:")
print(predicted_labels)

Actual Labels:
tensor([[0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.]])

Predicted Probabilities:
tensor([[4.1968e-05],
        [2.2245e-02],
        [8.8546e-01],
        [8.8546e-01],
        [9.7622e-04],
        [8.8546e-01]])

Predicted Labels:
tensor([[0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.]])


## Results and Conclusion

A feed-forward neural network was implemented using PyTorch for binary
classification of student performance.

### Results
- Input Features: Study Hours, Attendance, Previous Score
- Hidden Layers: 2
- Activation Function: ReLU
- Output Activation: Sigmoid
- Loss Function: Binary Cross-Entropy
- Optimizer: Adam
- Epochs: 200
- Final Training Loss: 0.1956
- Training Accuracy: 100%

The model correctly classified all 6 training samples after feature
normalization. Feature normalization helped improve the training process
and reduce the loss significantly.

Note: The dataset used for this experiment is very small, so the training
accuracy should not be considered a measure of real-world generalization.
